In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from itertools import chain

In [2]:
model_path = "/kaggle/input/notebooks/shipengalan/download-model/models/Qwen3.5-0.8B"
train_data_dir = "/kaggle/input/datasets/shipengalan/mydata/text/zh/train_data"
max_seq_length = 10
num_proc = 1

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

In [4]:
train_dataset = load_dataset("text", data_dir=train_data_dir, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
def tokenization(example):
    return tokenizer(example["text"])

def group_texts(examples):
    concat_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concat_examples["input_ids"])
    total_length = (total_length // max_seq_length) * max_seq_length
    result = {k: [t[i:i+max_seq_length] for i in range(0, total_length, max_seq_length)] for k, t in concat_examples.items()}
    result["labels"] = result["input_ids"].copy()
    return result

In [6]:
train_dataset = load_dataset("text", data_dir=train_data_dir, split="train")
train_dataset = train_dataset.map(tokenization, remove_columns=["text"], num_proc=num_proc)
train_dataset = train_dataset.map(group_texts, batched=True)

Map (num_proc=1):   0%|          | 0/127 [00:00<?, ? examples/s]

Map:   0%|          | 0/127 [00:00<?, ? examples/s]

In [7]:
train_dataset[3]

{'input_ids': [110699,
  96183,
  95923,
  100222,
  96826,
  95726,
  112832,
  99970,
  95849,
  3709],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [110699,
  96183,
  95923,
  100222,
  96826,
  95726,
  112832,
  99970,
  95849,
  3709]}

In [8]:
train_dataset[100]

{'input_ids': [114079,
  110404,
  3709,
  108547,
  96794,
  95913,
  128783,
  16487,
  101982,
  100750],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'labels': [114079,
  110404,
  3709,
  108547,
  96794,
  95913,
  128783,
  16487,
  101982,
  100750]}

In [9]:
print(tokenizer.bos_token_id, tokenizer.bos_token_id)

None None


In [10]:
tokenizer("我爱北京天安门")

{'input_ids': [102286, 98116, 128028], 'attention_mask': [1, 1, 1]}

In [11]:
tokenizer.decode([128028])

'天安门'